# Import

In [1]:
import pandas as pd
import json

from curation_tools.curation_tools import (
    CuratedDataset,
    ObsSchema,
    VarSchema,
    Experiment,
    download_file,
    upload_parquet_to_bq
)

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    handlers=[
        logging.FileHandler("curation.log"),
        logging.StreamHandler(),  # keep console output too
    ],
    force=True,
)
# show all columns
pd.set_option('display.max_columns', None)

/nfs/production/mfreeberg/perturb-seq/aleks/PerturbationCatalogue/.venv/lib/python3.12/site-packages/pandera/_pandas_deprecated.py:157: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)


# Download data

Download the data from AWS as such:

`!aws s3 cp --no-sign-request s3://genome-scale-tcell-perturb-seq/marson2025_data/{name_of_the_file}.h5ad ..aa/non_curated/h5ad/{name_of_the_file}.h5ad`


# Initialise the dataset object

In [2]:
noncurated_path = '../non_curated/h5ad/zhu_2025_D4_stim48hr_cl.h5ad'
cur_data = CuratedDataset(
    obs_schema=ObsSchema,
    var_schema=VarSchema,
    exp_metadata_schema=Experiment,
    noncurated_path=noncurated_path
)

cur_data.load_data()

Loading data from ../non_curated/h5ad/zhu_2025_D4_stim48hr_cl.h5ad


In [3]:
cur_data.adata.obs

,lane_id,n_genes_by_counts,total_counts,pct_counts_mt,top_guide_UMI_counts,guide_id,perturbed_gene_name,perturbed_gene_id,guide_type,PuroR,guide_group,low_quality
AAACAAGCAAACCCAAATACGTCA-1_CD4i_R2L41_CD4i_R2_D4_Stim48hr_CD4i_R2_Ultima,CD4i_R2L41,6739,33708.0,1.302361,1862.0,NDUFB5-1,NDUFB5,ENSG00000136521,targeting,3.264477,targeting single sgRNA,False
AAACAAGCAACAAAGAAACGCCGA-1_CD4i_R2L41_CD4i_R2_D4_Stim48hr_CD4i_R2_Ultima,CD4i_R2L41,5907,19897.0,0.638287,110.0,SERPINB9-1,SERPINB9,ENSG00000170542,targeting,2.623981,targeting single sgRNA,False
AAACAAGCAACGTTAGATCATGTG-1_CD4i_R2L41_CD4i_R2_D4_Stim48hr_CD4i_R2_Ultima,CD4i_R2L41,3868,9237.0,0.086608,92.0,ZNF33A-2,ZNF33A,ENSG00000189180,targeting,1.401497,targeting single sgRNA,False
AAACAAGCAACTAAGTAACGCCGA-1_CD4i_R2L41_CD4i_R2_D4_Stim48hr_CD4i_R2_Ultima,CD4i_R2L41,5914,20720.0,0.115830,275.0,LYG1-1,LYG1,ENSG00000144214,targeting,1.628095,targeting single sgRNA,False
AAACAAGCAACTAAGTATACGTCA-1_CD4i_R2L41_CD4i_R2_D4_Stim48hr_CD4i_R2_Ultima,CD4i_R2L41,5828,19086.0,1.152677,NaN,NaN,NaN,NaN,NaN,0.908886,no sgRNA,False
...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTGAGTTGTTGGTATTCGGTT-1_CD4i_R2L36_CD4i_R2_D4_Stim48hr_CD4i_R2_Ultima,CD4i_R2L36,4647,10932.0,0.146359,17.0,HIF1AN-1,HIF1AN,ENSG00000166135,targeting,4.133083,targeting single sgRNA,False
TTTGTGAGTTTACCGGATCATGTG-1_CD4i_R2L36_CD4i_R2_D4_Stim48hr_CD4i_R2_Ultima,CD4i_R2L36,3795,8155.0,0.147149,447.0,TXNRD2-2,TXNRD2,ENSG00000184470,targeting,2.579297,targeting single sgRNA,False
TTTGTGAGTTTACGACATACGTCA-1_CD4i_R2L36_CD4i_R2_D4_Stim48hr_CD4i_R2_Ultima,CD4i_R2L36,4135,10237.0,0.488424,169.0,ZYG11B-1,ZYG11B,ENSG00000162378,targeting,3.714791,targeting single sgRNA,False
TTTGTGAGTTTGCATAATACGTCA-1_CD4i_R2L36_CD4i_R2_D4_Stim48hr_CD4i_R2_Ultima,CD4i_R2L36,4897,13835.0,1.633538,71.0,PPP2R3C-1,PPP2R3C,ENSG00000092020,targeting,2.988620,targeting single sgRNA,False


In [4]:
cur_data.adata.var

,gene_ids,feature_types,genome,gene_name,mt
CUSTOM001_PuroR,CUSTOM001_PuroR,Gene Expression,GRCh38,PuroR,False
ENSG00000000003,ENSG00000000003,Gene Expression,GRCh38,TSPAN6,False
ENSG00000000005,ENSG00000000005,Gene Expression,GRCh38,TNMD,False
ENSG00000000419,ENSG00000000419,Gene Expression,GRCh38,DPM1,False
ENSG00000000457,ENSG00000000457,Gene Expression,GRCh38,SCYL3,False
...,...,...,...,...,...
ENSG00000291110,ENSG00000291110,Gene Expression,GRCh38,TRIM16L,False
ENSG00000291122,ENSG00000291122,Gene Expression,GRCh38,CASTOR3P,False
ENSG00000291135,ENSG00000291135,Gene Expression,GRCh38,FCGR1BP,False
ENSG00000291145,ENSG00000291145,Gene Expression,GRCh38,PPP5D1P,False


# OBS slot curation

### Since for multi-guide perturbations the identities of said guides are unknown, we are filtering out these cells.

In [5]:
print(f"Number of cells before filtering: {cur_data.adata.n_obs}")
cur_data.adata = cur_data.adata[cur_data.adata.obs['guide_id'] != 'multi_sgRNA']
print(f"Number of cells after filtering: {cur_data.adata.n_obs}")

Number of cells before filtering: 2815784
Number of cells after filtering: 2208902


### Add `index` as `perturbation_name`

In [6]:
cur_data.adata.obs['cell_barcode'] = cur_data.adata.obs.index.str.split('_').str[0]
cur_data.adata.obs['perturbation_name'] = cur_data.adata.obs['cell_barcode'] +'_'+cur_data.adata.obs['lane_id'].astype(str)

/tmp/ipykernel_129282/2231587657.py:1: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  cur_data.adata.obs['cell_barcode'] = cur_data.adata.obs.index.str.split('_').str[0]


### Show unique perturbations

In [ ]:
cur_data.show_unique(slot = 'obs', column = 'perturbation_name')

### Add guide RNA information

In [8]:
# download the guide RNA spreadsheet
download_file(
    url="https://raw.githubusercontent.com/emdann/GWT_perturbseq_analysis_2025/refs/heads/master/metadata/suppl_tables/sgrna_library_metadata.suppl_table.csv",
    dest_path="../supplementary/zhu_2025_guide_info.csv"
)
# read in the guide RNA info csv
guide_info_df = pd.read_csv("../supplementary/zhu_2025_guide_info.csv")

guide_info_df = guide_info_df[['sgRNA', 'seq']]

guide_info_df['sgRNA'] = guide_info_df['sgRNA'].str.replace('1-Jun','JUN-1').str.replace('2-Jun','JUN-2')

guide_info_df = guide_info_df.rename(columns={'seq': 'guide_sequence', 'sgRNA':'guide_id'})

guide_info_df


File ../supplementary/zhu_2025_guide_info.csv already exists. Skipping download.


,guide_id,guide_sequence
0,ARMC5-1,CTGCCTCGCGCAGCTCGCGG
1,DDB1-1,GGAGTTCGCTGCGCGCTGTT
2,FNDC10-1,TGCCCGCTCCCCGCGATCCC
3,RORC-1,TCTGTGGGGCCCTGTCCATG
4,SLC35F6-1,GAACAGCTGGTACTTGGTCC
...,...,...
26499,NTC-988,TTTCAGGCTACGGGCGCGGG
26500,NTC-989,TTTCCCATGATCATTTAGTG
26501,NTC-990,TTTCGCCCAAGAGGCTTGGG
26502,NTC-991,TTTCGTGCCGATGTAACACA


In [9]:
print("Number of overlapping guides with cur_data:")
cur_data.adata.obs['guide_id'].isin(guide_info_df['guide_id'].to_list()).value_counts()

Number of overlapping guides with cur_data:


guide_id
True     1888192
False     320710
Name: count, dtype: int64

In [10]:
# merge cur_data.adata.obs with guide_info_df
cur_data.adata.obs = cur_data.adata.obs.merge(guide_info_df, on='guide_id', how='left')
# check that there are no missing guide sequences
print(f"Number of missing guide sequences: {cur_data.adata.obs['guide_sequence'].isna().sum()}")


/homes/zakirov/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


Number of missing guide sequences: 320710


In [11]:
cur_data.adata.obs

,lane_id,n_genes_by_counts,total_counts,pct_counts_mt,top_guide_UMI_counts,guide_id,perturbed_gene_name,perturbed_gene_id,guide_type,PuroR,guide_group,low_quality,cell_barcode,perturbation_name,guide_sequence
0,CD4i_R2L41,6739,33708.0,1.302361,1862.0,NDUFB5-1,NDUFB5,ENSG00000136521,targeting,3.264477,targeting single sgRNA,False,AAACAAGCAAACCCAAATACGTCA-1,AAACAAGCAAACCCAAATACGTCA-1_CD4i_R2L41,GCGGCGGGTTTCGGTTACTG
1,CD4i_R2L41,5907,19897.0,0.638287,110.0,SERPINB9-1,SERPINB9,ENSG00000170542,targeting,2.623981,targeting single sgRNA,False,AAACAAGCAACAAAGAAACGCCGA-1,AAACAAGCAACAAAGAAACGCCGA-1_CD4i_R2L41,TCCAGGCTCGGAGCGCGGCA
2,CD4i_R2L41,3868,9237.0,0.086608,92.0,ZNF33A-2,ZNF33A,ENSG00000189180,targeting,1.401497,targeting single sgRNA,False,AAACAAGCAACGTTAGATCATGTG-1,AAACAAGCAACGTTAGATCATGTG-1_CD4i_R2L41,GAGGCCCGGGGACCTCATAG
3,CD4i_R2L41,5914,20720.0,0.115830,275.0,LYG1-1,LYG1,ENSG00000144214,targeting,1.628095,targeting single sgRNA,False,AAACAAGCAACTAAGTAACGCCGA-1,AAACAAGCAACTAAGTAACGCCGA-1_CD4i_R2L41,GAGGCATGGATCTTACACTC
4,CD4i_R2L41,5828,19086.0,1.152677,NaN,NaN,NaN,NaN,NaN,0.908886,no sgRNA,False,AAACAAGCAACTAAGTATACGTCA-1,AAACAAGCAACTAAGTATACGTCA-1_CD4i_R2L41,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2208897,CD4i_R2L36,4647,10932.0,0.146359,17.0,HIF1AN-1,HIF1AN,ENSG00000166135,targeting,4.133083,targeting single sgRNA,False,TTTGTGAGTTGTTGGTATTCGGTT-1,TTTGTGAGTTGTTGGTATTCGGTT-1_CD4i_R2L36,AGGGCTCCAGCCTCCTCCCG
2208898,CD4i_R2L36,3795,8155.0,0.147149,447.0,TXNRD2-2,TXNRD2,ENSG00000184470,targeting,2.579297,targeting single sgRNA,False,TTTGTGAGTTTACCGGATCATGTG-1,TTTGTGAGTTTACCGGATCATGTG-1_CD4i_R2L36,GGGCGCTGCGGGGATTAGGA
2208899,CD4i_R2L36,4135,10237.0,0.488424,169.0,ZYG11B-1,ZYG11B,ENSG00000162378,targeting,3.714791,targeting single sgRNA,False,TTTGTGAGTTTACGACATACGTCA-1,TTTGTGAGTTTACGACATACGTCA-1_CD4i_R2L36,GCTCCTAGCTTGAGGGAAAG
2208900,CD4i_R2L36,4897,13835.0,1.633538,71.0,PPP2R3C-1,PPP2R3C,ENSG00000092020,targeting,2.988620,targeting single sgRNA,False,TTTGTGAGTTTGCATAATACGTCA-1,TTTGTGAGTTTGCATAATACGTCA-1_CD4i_R2L36,GCCCGGCCCTAAACTCCTAG


In [12]:
# change perturbed_gene_id from categorical to string
cur_data.adata.obs['perturbed_gene_id'] = cur_data.adata.obs['perturbed_gene_id'].astype(str)
cur_data.adata.obs['perturbed_gene_name'] = cur_data.adata.obs['perturbed_gene_name'].astype(str)

cur_data.adata.obs.loc[cur_data.adata.obs['perturbed_gene_id'].isin(['NTC']), 'perturbed_gene_id'] = 'control_nontargeting'
cur_data.adata.obs.loc[cur_data.adata.obs['perturbed_gene_name'].isin(['NTC']), 'perturbed_gene_name'] = 'control_nontargeting'

cur_data.adata.obs.loc[cur_data.adata.obs['perturbed_gene_id'].isna(), 'perturbed_gene_id'] = 'control_casonly'
cur_data.adata.obs.loc[cur_data.adata.obs['perturbed_gene_name'].isna(), 'perturbed_gene_name'] = 'control_casonly'

### Standardise perturbation targets

In [13]:
cur_data.standardize_genes(
    slot='obs',
    input_column='perturbed_gene_id',
    input_column_type='ensembl_gene_id',
    multiple_entries=False,
    # remove_version=True,
    # version_sep='.'
)

Missing Ensembl IDs: ['ENSG00000182230', 'ENSG00000277796', 'ENSG00000148362', 'SGK494', 'LUZP6', 'ENSG00000256618', 'ENSG00000232196', 'SEPT6', 'U2AF1L5', 'ENSG00000203812', 'ENSG00000255823', 'AKAP2', 'OCLM', 'SMIM11B']; attempting to fetch latest IDs...
Fetched latest Ensembl IDs: {'ENSG00000182230': nan, 'ENSG00000277796': 'ENSG00000293545', 'ENSG00000148362': 'ENSG00000310560', 'ENSG00000256618': nan, 'ENSG00000232196': nan, 'ENSG00000203812': 'ENSG00000288825', 'ENSG00000255823': nan}
--------------------------------------------------
Successfully mapped 12423 out of 12423 Ensembl IDs.
--------------------------------------------------


/homes/zakirov/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [14]:
cur_data.adata.obs

,guide_group,n_genes_by_counts,guide_type,low_quality,lane_id,guide_sequence,guide_id,perturbed_gene_id,top_guide_UMI_counts,perturbed_gene_name,cell_barcode,total_counts,perturbation_name,pct_counts_mt,PuroR,perturbed_target_ensg,perturbed_target_symbol,perturbed_target_biotype,perturbed_target_coord,perturbed_target_chromosome,original_index
index,,,,,,,,,,,,,,,,,,,,,
0,targeting single sgRNA,6739,targeting,False,CD4i_R2L41,GCGGCGGGTTTCGGTTACTG,NDUFB5-1,ENSG00000136521,1862.0,NDUFB5,AAACAAGCAAACCCAAATACGTCA-1,33708.0,AAACAAGCAAACCCAAATACGTCA-1_CD4i_R2L41,1.302361,3.264477,ENSG00000136521,NDUFB5,protein_coding,chr3:179604690-179627647;1,3,0
1,targeting single sgRNA,5907,targeting,False,CD4i_R2L41,TCCAGGCTCGGAGCGCGGCA,SERPINB9-1,ENSG00000170542,110.0,SERPINB9,AAACAAGCAACAAAGAAACGCCGA-1,19897.0,AAACAAGCAACAAAGAAACGCCGA-1_CD4i_R2L41,0.638287,2.623981,ENSG00000170542,SERPINB9,protein_coding,chr6:2887169-2903309;-1,6,1
2,targeting single sgRNA,3868,targeting,False,CD4i_R2L41,GAGGCCCGGGGACCTCATAG,ZNF33A-2,ENSG00000189180,92.0,ZNF33A,AAACAAGCAACGTTAGATCATGTG-1,9237.0,AAACAAGCAACGTTAGATCATGTG-1_CD4i_R2L41,0.086608,1.401497,ENSG00000189180,ZNF33A,protein_coding,chr10:38010295-38065088;1,10,2
3,targeting single sgRNA,5914,targeting,False,CD4i_R2L41,GAGGCATGGATCTTACACTC,LYG1-1,ENSG00000144214,275.0,LYG1,AAACAAGCAACTAAGTAACGCCGA-1,20720.0,AAACAAGCAACTAAGTAACGCCGA-1_CD4i_R2L41,0.115830,1.628095,ENSG00000144214,LYG1,protein_coding,chr2:99284238-99306110;-1,2,3
4,no sgRNA,5828,NaN,False,CD4i_R2L41,NaN,NaN,nan,NaN,nan,AAACAAGCAACTAAGTATACGTCA-1,19086.0,AAACAAGCAACTAAGTATACGTCA-1_CD4i_R2L41,1.152677,0.908886,nan,NaN,NaN,NaN,NaN,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2208897,targeting single sgRNA,4647,targeting,False,CD4i_R2L36,AGGGCTCCAGCCTCCTCCCG,HIF1AN-1,ENSG00000166135,17.0,HIF1AN,TTTGTGAGTTGTTGGTATTCGGTT-1,10932.0,TTTGTGAGTTGTTGGTATTCGGTT-1_CD4i_R2L36,0.146359,4.133083,ENSG00000166135,HIF1AN,protein_coding,chr10:100529072-100559998;1,10,2208897
2208898,targeting single sgRNA,3795,targeting,False,CD4i_R2L36,GGGCGCTGCGGGGATTAGGA,TXNRD2-2,ENSG00000184470,447.0,TXNRD2,TTTGTGAGTTTACCGGATCATGTG-1,8155.0,TTTGTGAGTTTACCGGATCATGTG-1_CD4i_R2L36,0.147149,2.579297,ENSG00000184470,TXNRD2,protein_coding,chr22:19875517-19941820;-1,22,2208898
2208899,targeting single sgRNA,4135,targeting,False,CD4i_R2L36,GCTCCTAGCTTGAGGGAAAG,ZYG11B-1,ENSG00000162378,169.0,ZYG11B,TTTGTGAGTTTACGACATACGTCA-1,10237.0,TTTGTGAGTTTACGACATACGTCA-1_CD4i_R2L36,0.488424,3.714791,ENSG00000162378,ZYG11B,protein_coding,chr1:52726447-52827336;1,1,2208899


# Manually replace some genes

In [15]:
# define genes to replace in the cur_data.adata.obs
genes_to_replace = cur_data.adata.obs[cur_data.adata.obs['perturbed_target_symbol'].isna()]['perturbed_gene_name'].drop_duplicates().to_list()
# define columns to replace
cols_to_replace = ['perturbed_target_ensg',	'perturbed_target_symbol',	'perturbed_target_biotype',	'perturbed_target_coord',	'perturbed_target_chromosome']
# define mapping dict for renaming columns in gene_ont for replacement
gene_ont_col_mapping = {
    'ensembl_gene_id':'perturbed_target_ensg',
    'gene_symbol':'perturbed_target_symbol',
    'biotype':'perturbed_target_biotype',
    'gene_coord':'perturbed_target_coord',
    'chromosome_name':'perturbed_target_chromosome'
}
# loop through genes and fill in the missing cols
for replacement_gene in genes_to_replace:
    if replacement_gene in cur_data.gene_ont['synonym'].values:
        # subset gene_ont and get values for replacement
        replacement = (
            cur_data.gene_ont.rename(columns=gene_ont_col_mapping)
            .loc[cur_data.gene_ont['synonym'].isin([replacement_gene]), cols_to_replace]
        )[cols_to_replace].values[0]
        # replace values in cur_data.adata.obs with the values from gene_ont
        cur_data.adata.obs.loc[cur_data.adata.obs['perturbed_gene_name'].isin([replacement_gene]), cols_to_replace] = replacement
        print(f"Replaced {replacement_gene} with {replacement}")
    else:
        print(f"No replacement found for {replacement_gene}")


No replacement found for nan
No replacement found for SMIM11B
No replacement found for AKAP2
No replacement found for OCLM
Replaced SGK494 with ['ENSG00000167524' 'RSKR' 'protein_coding' 'chr17:28455752-28614197;-1'
 '17']
No replacement found for LUZP6
Replaced SEPT6 with ['ENSG00000125354' 'SEPTIN6' 'protein_coding'
 'chrX:119615724-119693949;-1' 'X']
No replacement found for FAM153B
No replacement found for CCL3L1
No replacement found for MTRNR2L4
No replacement found for MTRNR2L8
No replacement found for MTRNR2L1
No replacement found for U2AF1L5


In [16]:
# replace non-mapped perturbed_target_symbol with original gene symbols
cur_data.adata.obs.loc[cur_data.adata.obs['perturbed_target_symbol'].isna(), 'perturbed_target_symbol'] = cur_data.adata.obs.loc[cur_data.adata.obs['perturbed_target_symbol'].isna(), 'perturbed_gene_name'].values

In [17]:
# replace entries in perturbed_target_ensg that are not ENSEMBL ids with NA
cur_data.adata.obs.loc[(~cur_data.adata.obs['perturbed_target_ensg'].str.startswith('ENSG')) & (cur_data.adata.obs['perturbed_target_ensg'] != 'control_nontargeting'), 'perturbed_target_ensg'] = pd.NA

In [18]:
cur_data.adata.obs

,guide_group,n_genes_by_counts,guide_type,low_quality,lane_id,guide_sequence,guide_id,perturbed_gene_id,top_guide_UMI_counts,perturbed_gene_name,cell_barcode,total_counts,perturbation_name,pct_counts_mt,PuroR,perturbed_target_ensg,perturbed_target_symbol,perturbed_target_biotype,perturbed_target_coord,perturbed_target_chromosome,original_index
index,,,,,,,,,,,,,,,,,,,,,
0,targeting single sgRNA,6739,targeting,False,CD4i_R2L41,GCGGCGGGTTTCGGTTACTG,NDUFB5-1,ENSG00000136521,1862.0,NDUFB5,AAACAAGCAAACCCAAATACGTCA-1,33708.0,AAACAAGCAAACCCAAATACGTCA-1_CD4i_R2L41,1.302361,3.264477,ENSG00000136521,NDUFB5,protein_coding,chr3:179604690-179627647;1,3,0
1,targeting single sgRNA,5907,targeting,False,CD4i_R2L41,TCCAGGCTCGGAGCGCGGCA,SERPINB9-1,ENSG00000170542,110.0,SERPINB9,AAACAAGCAACAAAGAAACGCCGA-1,19897.0,AAACAAGCAACAAAGAAACGCCGA-1_CD4i_R2L41,0.638287,2.623981,ENSG00000170542,SERPINB9,protein_coding,chr6:2887169-2903309;-1,6,1
2,targeting single sgRNA,3868,targeting,False,CD4i_R2L41,GAGGCCCGGGGACCTCATAG,ZNF33A-2,ENSG00000189180,92.0,ZNF33A,AAACAAGCAACGTTAGATCATGTG-1,9237.0,AAACAAGCAACGTTAGATCATGTG-1_CD4i_R2L41,0.086608,1.401497,ENSG00000189180,ZNF33A,protein_coding,chr10:38010295-38065088;1,10,2
3,targeting single sgRNA,5914,targeting,False,CD4i_R2L41,GAGGCATGGATCTTACACTC,LYG1-1,ENSG00000144214,275.0,LYG1,AAACAAGCAACTAAGTAACGCCGA-1,20720.0,AAACAAGCAACTAAGTAACGCCGA-1_CD4i_R2L41,0.115830,1.628095,ENSG00000144214,LYG1,protein_coding,chr2:99284238-99306110;-1,2,3
4,no sgRNA,5828,NaN,False,CD4i_R2L41,NaN,NaN,nan,NaN,nan,AAACAAGCAACTAAGTATACGTCA-1,19086.0,AAACAAGCAACTAAGTATACGTCA-1_CD4i_R2L41,1.152677,0.908886,<NA>,nan,NaN,NaN,NaN,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2208897,targeting single sgRNA,4647,targeting,False,CD4i_R2L36,AGGGCTCCAGCCTCCTCCCG,HIF1AN-1,ENSG00000166135,17.0,HIF1AN,TTTGTGAGTTGTTGGTATTCGGTT-1,10932.0,TTTGTGAGTTGTTGGTATTCGGTT-1_CD4i_R2L36,0.146359,4.133083,ENSG00000166135,HIF1AN,protein_coding,chr10:100529072-100559998;1,10,2208897
2208898,targeting single sgRNA,3795,targeting,False,CD4i_R2L36,GGGCGCTGCGGGGATTAGGA,TXNRD2-2,ENSG00000184470,447.0,TXNRD2,TTTGTGAGTTTACCGGATCATGTG-1,8155.0,TTTGTGAGTTTACCGGATCATGTG-1_CD4i_R2L36,0.147149,2.579297,ENSG00000184470,TXNRD2,protein_coding,chr22:19875517-19941820;-1,22,2208898
2208899,targeting single sgRNA,4135,targeting,False,CD4i_R2L36,GCTCCTAGCTTGAGGGAAAG,ZYG11B-1,ENSG00000162378,169.0,ZYG11B,TTTGTGAGTTTACGACATACGTCA-1,10237.0,TTTGTGAGTTTACGACATACGTCA-1_CD4i_R2L36,0.488424,3.714791,ENSG00000162378,ZYG11B,protein_coding,chr1:52726447-52827336;1,1,2208899


### Add `perturbed_target_number` column

In [19]:
# cur_data.count_entries(
#     slot='obs',
#     input_column='perturbed_target_ensg',
#     count_column_name='perturbed_target_number',
#     sep='|'
# )
cur_data.adata.obs['perturbed_target_number'] = 1

### Encode chromosomes as integers

In [20]:
cur_data.chromosome_encoding()

Chromosome encoding applied to perturbed_target_chromosome in adata.obs and stored as 'perturbed_target_chromosome_encoding'.


In [21]:
cur_data.adata.obs[['perturbation_name', 'perturbed_target_chromosome_encoding']]

,perturbation_name,perturbed_target_chromosome_encoding
index,,
0,AAACAAGCAAACCCAAATACGTCA-1_CD4i_R2L41,3
1,AAACAAGCAACAAAGAAACGCCGA-1_CD4i_R2L41,6
2,AAACAAGCAACGTTAGATCATGTG-1_CD4i_R2L41,10
3,AAACAAGCAACTAAGTAACGCCGA-1_CD4i_R2L41,2
4,AAACAAGCAACTAAGTATACGTCA-1_CD4i_R2L41,0
...,...,...
2208897,TTTGTGAGTTGTTGGTATTCGGTT-1_CD4i_R2L36,10
2208898,TTTGTGAGTTTACCGGATCATGTG-1_CD4i_R2L36,22
2208899,TTTGTGAGTTTACGACATACGTCA-1_CD4i_R2L36,1


In [22]:
cur_data.adata.obs

,guide_group,n_genes_by_counts,guide_type,low_quality,lane_id,guide_sequence,guide_id,perturbed_gene_id,top_guide_UMI_counts,perturbed_gene_name,cell_barcode,total_counts,perturbation_name,pct_counts_mt,PuroR,perturbed_target_ensg,perturbed_target_symbol,perturbed_target_biotype,perturbed_target_coord,perturbed_target_chromosome,original_index,perturbed_target_number,perturbed_target_chromosome_encoding
index,,,,,,,,,,,,,,,,,,,,,,,
0,targeting single sgRNA,6739,targeting,False,CD4i_R2L41,GCGGCGGGTTTCGGTTACTG,NDUFB5-1,ENSG00000136521,1862.0,NDUFB5,AAACAAGCAAACCCAAATACGTCA-1,33708.0,AAACAAGCAAACCCAAATACGTCA-1_CD4i_R2L41,1.302361,3.264477,ENSG00000136521,NDUFB5,protein_coding,chr3:179604690-179627647;1,3,0,1,3
1,targeting single sgRNA,5907,targeting,False,CD4i_R2L41,TCCAGGCTCGGAGCGCGGCA,SERPINB9-1,ENSG00000170542,110.0,SERPINB9,AAACAAGCAACAAAGAAACGCCGA-1,19897.0,AAACAAGCAACAAAGAAACGCCGA-1_CD4i_R2L41,0.638287,2.623981,ENSG00000170542,SERPINB9,protein_coding,chr6:2887169-2903309;-1,6,1,1,6
2,targeting single sgRNA,3868,targeting,False,CD4i_R2L41,GAGGCCCGGGGACCTCATAG,ZNF33A-2,ENSG00000189180,92.0,ZNF33A,AAACAAGCAACGTTAGATCATGTG-1,9237.0,AAACAAGCAACGTTAGATCATGTG-1_CD4i_R2L41,0.086608,1.401497,ENSG00000189180,ZNF33A,protein_coding,chr10:38010295-38065088;1,10,2,1,10
3,targeting single sgRNA,5914,targeting,False,CD4i_R2L41,GAGGCATGGATCTTACACTC,LYG1-1,ENSG00000144214,275.0,LYG1,AAACAAGCAACTAAGTAACGCCGA-1,20720.0,AAACAAGCAACTAAGTAACGCCGA-1_CD4i_R2L41,0.115830,1.628095,ENSG00000144214,LYG1,protein_coding,chr2:99284238-99306110;-1,2,3,1,2
4,no sgRNA,5828,NaN,False,CD4i_R2L41,NaN,NaN,nan,NaN,nan,AAACAAGCAACTAAGTATACGTCA-1,19086.0,AAACAAGCAACTAAGTATACGTCA-1_CD4i_R2L41,1.152677,0.908886,<NA>,nan,NaN,NaN,NaN,4,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2208897,targeting single sgRNA,4647,targeting,False,CD4i_R2L36,AGGGCTCCAGCCTCCTCCCG,HIF1AN-1,ENSG00000166135,17.0,HIF1AN,TTTGTGAGTTGTTGGTATTCGGTT-1,10932.0,TTTGTGAGTTGTTGGTATTCGGTT-1_CD4i_R2L36,0.146359,4.133083,ENSG00000166135,HIF1AN,protein_coding,chr10:100529072-100559998;1,10,2208897,1,10
2208898,targeting single sgRNA,3795,targeting,False,CD4i_R2L36,GGGCGCTGCGGGGATTAGGA,TXNRD2-2,ENSG00000184470,447.0,TXNRD2,TTTGTGAGTTTACCGGATCATGTG-1,8155.0,TTTGTGAGTTTACCGGATCATGTG-1_CD4i_R2L36,0.147149,2.579297,ENSG00000184470,TXNRD2,protein_coding,chr22:19875517-19941820;-1,22,2208898,1,22
2208899,targeting single sgRNA,4135,targeting,False,CD4i_R2L36,GCTCCTAGCTTGAGGGAAAG,ZYG11B-1,ENSG00000162378,169.0,ZYG11B,TTTGTGAGTTTACGACATACGTCA-1,10237.0,TTTGTGAGTTTACGACATACGTCA-1_CD4i_R2L36,0.488424,3.714791,ENSG00000162378,ZYG11B,protein_coding,chr1:52726447-52827336;1,1,2208899,1,1


### Curate replicates

In [23]:
cur_data.adata.obs = cur_data.adata.obs.rename(columns={'lane_id':'technical_replicate'})

### Add metadata

In [24]:
cur_data.create_columns(
    overwrite=True,
    slot="obs",
    col_dict={
        "dataset_id": cur_data.dataset_id,
        "sample_id": range(1, cur_data.adata.obs.shape[0] + 1),
        # perturbation type
        "perturbation_type_label": "CRISPRi",
        "perturbation_type_id": None,
        "data_modality": "Perturb-seq",
        "significant": None,
        "significance_criteria": None,
        "score_interpretation": None,

        # treatment
        "treatment_label": "anti-CD3 antibody|anti-CD28 antibody|anti-CD2 antibody",
        "treatment_id": "EFO:0003317|EFO:0003304|NCIT:C184729",
        # replicates
        # "technical_replicate": None,
        "biological_replicate": "D4_CE0006864",
        # model system
        "model_system_label": "primary_cell",
        "model_system_id": None,
        "tissue": "lymphoid tissue",
        "cell_line_label": None,
        "cell_line_id": None,
        "cell_type_label": "CD4-positive, alpha-beta T cell",
        "disease_label": "healthy",
        "disease_id": None,

        "timepoint": "P14DT0H0M0S",
        "species": "Homo sapiens",
        "sex_label": "male",
        "sex_id": None,
        "developmental_stage_label": "adult",
        "developmental_stage_id": None,

        "study_title": "Genome-scale perturb-seq in primary human CD4+ T cells maps context-specific regulators of T cell programs and human immune traits",
        "study_uri": "https://doi.org/10.64898/2025.12.23.696273",
        "study_year": 2025,
        "first_author": "Ronghui Zhu",
        "last_author": "Alexander Marson",

        "experiment_title": "Perturb-seq of primary human CD4-positive T cells for patient D4_CE0006864, stimulated for 48hr with ImmunoCult CD3/CD28/CD2 activator",
        "experiment_summary": """
            Isolated human CD4-positive cells from four healthy donors were stimulated with ImmunoCult CD3/CD28/CD2 activator and sequentially transduced with dCas9-KRAB-Zim3 lentivirus (next morning after stimulation) and a Perturb-seq guide library (next afternoon after stimulation, MOI 0.2).
            The library consisted all genes expressed in human CD4+ T cells, all transcription factors annotated in the Lambert et al. (2018) and non-targeting controls totalling 12,748 genes.
            The gRNA sequences were selected from hCRISPRiv2 and Dolcetto libraries.
            On day 12, cells were split into three conditions:
            (1) Rest - cells were left for 8 hr without stimulation;
            (2) Stim8hr - cells were stimulated with ImmunoCult CD3/CD28/CD2 activator for 8 hr;
            (3) Stim48hr - cells were stimulated with ImmunoCult CD3/CD28/CD2 activator for 48 hr.
            Cells were harvested, fixed, stored using GEM-X Flex Sample Preparation v2 Kit and sequenced using GEM-X Flex Gene Expression Human n-plex kit converted into Ultima compatible libraries for sequencing on the Ultima Genomics UG100.
            """,

        "number_of_perturbed_targets": len(set(cur_data.adata.obs['perturbed_target_symbol'])),
        "number_of_perturbed_samples": cur_data.adata.obs.shape[0],

        "library_generation_type_id": "EFO:0022868",
        "library_generation_type_label": "endogenous",

        "library_generation_method_id": None,
        "library_generation_method_label": "dCas9-KRAB-Zim3",

        "enzyme_delivery_method_id": None,
        "enzyme_delivery_method_label": "lentivirus transduction",

        "library_delivery_method_id": None,
        "library_delivery_method_label": "lentivirus transduction",

        "enzyme_integration_state_id": None,
        "enzyme_integration_state_label": "random locus integration",

        "library_integration_state_id": None,
        "library_integration_state_label": "random locus integration",

        "enzyme_expression_control_id": None,
        "enzyme_expression_control_label": "constitutive transgene expression",

        "library_expression_control_id": None,
        "library_expression_control_label": "constitutive transgene expression",

        "library_name": "custom",
        "library_uri": None,

        "library_format_id": None,
        "library_format_label": "pooled",

        "library_scope_id": None,
        "library_scope_label": "focused",

        "library_perturbation_type_id": None,
        "library_perturbation_type_label": "inhibition",

        "library_manufacturer": "Marson lab",
        "library_lentiviral_generation": "2",
        "library_grnas_per_target": "2",
        "library_total_grnas": str(cur_data.adata.obs['guide_sequence'].str.split('|').explode().nunique()),
        "library_total_variants": None,

        "readout_dimensionality_id": None,
        "readout_dimensionality_label": "high-dimensional assay",

        "readout_type_id": None,
        "readout_type_label": "transcriptomic",

        "readout_technology_id": None,
        "readout_technology_label": "single-cell rna-seq",

        "method_name_id": None,
        "method_name_label": "Perturb-seq",

        "method_uri": None,

        "sequencing_library_kit_id": None,
        "sequencing_library_kit_label": "GEM-X Flex Gene Expression Human n-plex kit",

        "sequencing_platform_id": None,
        "sequencing_platform_label": "Ultima Genomics UG100",

        "sequencing_strategy_id": None,
        "sequencing_strategy_label": "barcode sequencing",

        "software_counts_id": None,
        "software_counts_label": "CellRanger",

        "software_analysis_id": None,
        "software_analysis_label": "scanpy",

        "reference_genome_id": None,
        "reference_genome_label": "GRCh38",

        "license_label": "MIT License",
        "license_id": "SWO:9000074",

        "associated_datasets": json.dumps([
            {
                "dataset_accession": "Primary Human CD4+ T Cell Perturb-seq",
                "dataset_uri": "s3://genome-scale-tcell-perturb-seq/marson2025_data/D4_Stim48hr.assigned_guide.h5ad",
                "dataset_description": "Cell expression profiles for cells from donor D4_CE0006864, stimulated for 48hr with ImmunoCult CD3/CD28/CD2 activator.",
                "dataset_file_name": "D4_Stim48hr.assigned_guide.h5ad",
            }
        ])
    }
)

Column dataset_id added to adata.obs
Column sample_id added to adata.obs
Column perturbation_type_label added to adata.obs
Column perturbation_type_id added to adata.obs
Column data_modality added to adata.obs
Column significant added to adata.obs
Column significance_criteria added to adata.obs
Column score_interpretation added to adata.obs
Column treatment_label added to adata.obs
Column treatment_id added to adata.obs
Column biological_replicate added to adata.obs
Column model_system_label added to adata.obs
Column model_system_id added to adata.obs
Column tissue added to adata.obs
Column cell_line_label added to adata.obs
Column cell_line_id added to adata.obs
Column cell_type_label added to adata.obs
Column disease_label added to adata.obs
Column disease_id added to adata.obs
Column timepoint added to adata.obs
Column species added to adata.obs
Column sex_label added to adata.obs
Column sex_id added to adata.obs
Column developmental_stage_label added to adata.obs
Column development

In [25]:
cur_data.adata.obs

,guide_group,n_genes_by_counts,guide_type,low_quality,technical_replicate,guide_sequence,guide_id,perturbed_gene_id,top_guide_UMI_counts,perturbed_gene_name,cell_barcode,total_counts,perturbation_name,pct_counts_mt,PuroR,perturbed_target_ensg,perturbed_target_symbol,perturbed_target_biotype,perturbed_target_coord,perturbed_target_chromosome,original_index,perturbed_target_number,perturbed_target_chromosome_encoding,dataset_id,sample_id,perturbation_type_label,perturbation_type_id,data_modality,significant,significance_criteria,score_interpretation,treatment_label,treatment_id,biological_replicate,model_system_label,model_system_id,tissue,cell_line_label,cell_line_id,cell_type_label,disease_label,disease_id,timepoint,species,sex_label,sex_id,developmental_stage_label,developmental_stage_id,study_title,study_uri,study_year,first_author,last_author,experiment_title,experiment_summary,number_of_perturbed_targets,number_of_perturbed_samples,library_generation_type_id,library_generation_type_label,library_generation_method_id,library_generation_method_label,enzyme_delivery_method_id,enzyme_delivery_method_label,library_delivery_method_id,library_delivery_method_label,enzyme_integration_state_id,enzyme_integration_state_label,library_integration_state_id,library_integration_state_label,enzyme_expression_control_id,enzyme_expression_control_label,library_expression_control_id,library_expression_control_label,library_name,library_uri,library_format_id,library_format_label,library_scope_id,library_scope_label,library_perturbation_type_id,library_perturbation_type_label,library_manufacturer,library_lentiviral_generation,library_grnas_per_target,library_total_grnas,library_total_variants,readout_dimensionality_id,readout_dimensionality_label,readout_type_id,readout_type_label,readout_technology_id,readout_technology_label,method_name_id,method_name_label,method_uri,sequencing_library_kit_id,sequencing_library_kit_label,sequencing_platform_id,sequencing_platform_label,sequencing_strategy_id,sequencing_strategy_label,software_counts_id,software_counts_label,software_analysis_id,software_analysis_label,reference_genome_id,reference_genome_label,license_label,license_id,associated_datasets
index,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,targeting single sgRNA,6739,targeting,False,CD4i_R2L41,GCGGCGGGTTTCGGTTACTG,NDUFB5-1,ENSG00000136521,1862.0,NDUFB5,AAACAAGCAAACCCAAATACGTCA-1,33708.0,AAACAAGCAAACCCAAATACGTCA-1_CD4i_R2L41,1.302361,3.264477,ENSG00000136521,NDUFB5,protein_coding,chr3:179604690-179627647;1,3,0,1,3,zhu_2025_D4_stim48hr_cl,1,CRISPRi,None,Perturb-seq,None,None,None,anti-CD3 antibody|anti-CD28 antibody|anti-CD2 ...,EFO:0003317|EFO:0003304|NCIT:C184729,D4_CE0006864,primary_cell,None,lymphoid tissue,None,None,"CD4-positive, alpha-beta T cell",healthy,None,P14DT0H0M0S,Homo sapiens,male,None,adult,None,Genome-scale perturb-seq in primary human CD4+...,https://doi.org/10.64898/2025.12.23.696273,2025,Ronghui Zhu,Alexander Marson,Perturb-seq of primary human CD4-positive T ce...,\n Isolated human CD4-positive cell...,12423,2208902,EFO:0022868,endogenous,None,dCas9-KRAB-Zim3,None,lentivirus transduction,None,lentivirus transduction,None,random locus integration,None,random locus integration,None,constitutive transgene expression,None,constitutive transgene expression,custom,None,None,pooled,None,focused,None,inhibition,Marson lab,2,2,24803,None,None,high-dimensional assay,None,transcriptomic,None,single-cell rna-seq,None,Perturb-seq,None,None,GEM-X Flex Gene Expression Human n-plex kit,None,Ultima Genomics UG100,None,barcode sequencing,None,CellRanger,None,scanpy,None,GRCh38,MIT License,SWO:9000074,"[{""dataset_accession"": ""Primary Human CD4+ T C..."
1,targeting single sgRNA,5907,targeting,False,CD4i_R2L41,TCCAGGCTCGGAGCGCGGCA,SERPINB9-1,ENSG00000170542,110.0,SERPINB9,AAACAAGCAACAAAGAAACGCCGA-1,19897.0,AAACAAGCAACAAAGAAACGCCGA-1_CD4i_R2L41,0.638287,2.623981,ENSG000001705

### Curate tissue information


In [26]:
cur_data.standardize_ontology(
    input_column='tissue',
    column_type='term_name',
    ontology_type='tissue',
    overwrite=True
)

Mapped 1 tissue ontology terms from `tissue` column to ontology terms
DataFrame shape: (1, 4)
--------------------------------------------------
      input_column input_column_lower       name_lower     ontology_id
0  lymphoid tissue    lymphoid tissue  lymphoid tissue  UBERON:0001744
--------------------------------------------------


/homes/zakirov/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


### Curate cell type information

In [27]:
cur_data.standardize_ontology(
    input_column='cell_type_label',
    column_type='term_name',
    ontology_type='cell_type',
    overwrite=True
)

Mapped 1 cell_type ontology terms from `cell_type_label` column to ontology terms
DataFrame shape: (1, 4)
--------------------------------------------------
                      input_column               input_column_lower  \
0  CD4-positive, alpha-beta T cell  cd4-positive, alpha-beta t cell   

                        name_lower ontology_id  
0  cd4-positive, alpha-beta t cell  CL:0000624  
--------------------------------------------------
Overwriting column cell_type_label in adata.obs


/homes/zakirov/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


### Curate disease information

In [28]:
cur_data.standardize_ontology(
    input_column='disease_label',
    column_type='term_name',
    ontology_type='disease',
    overwrite=True
)

### Match schema column order

In [29]:
cur_data.match_schema_columns(slot='obs')

Matched columns of adata.obs to the obs_schema.


### Validate obs metadata

In [30]:
cur_data.validate_data(slot='obs', verbose=True)

2026-02-18 10:57:03,060 INFO curation_tools.curation_tools: adata.obs is valid according to the obs_schema.


,dataset_id,sample_id,data_modality,significant,significance_criteria,perturbation_name,perturbed_target_coord,perturbed_target_chromosome,perturbed_target_chromosome_encoding,perturbed_target_number,perturbed_target_ensg,perturbed_target_symbol,perturbed_target_biotype,guide_sequence,perturbation_type_label,perturbation_type_id,timepoint,treatment_label,treatment_id,technical_replicate,biological_replicate,model_system_label,model_system_id,species,tissue_label,tissue_id,cell_type_label,cell_type_id,cell_line_label,cell_line_id,sex_label,sex_id,developmental_stage_label,developmental_stage_id,disease_label,disease_id,study_title,study_uri,study_year,first_author,last_author,experiment_title,experiment_summary,number_of_perturbed_targets,number_of_perturbed_samples,library_generation_type_id,library_generation_type_label,library_generation_method_id,library_generation_method_label,enzyme_delivery_method_id,enzyme_delivery_method_label,library_delivery_method_id,library_delivery_method_label,enzyme_integration_state_id,enzyme_integration_state_label,library_integration_state_id,library_integration_state_label,enzyme_expression_control_id,enzyme_expression_control_label,library_expression_control_id,library_expression_control_label,library_name,library_uri,library_format_id,library_format_label,library_scope_id,library_scope_label,library_perturbation_type_id,library_perturbation_type_label,library_manufacturer,library_lentiviral_generation,library_grnas_per_target,library_total_grnas,library_total_variants,readout_dimensionality_id,readout_dimensionality_label,readout_type_id,readout_type_label,readout_technology_id,readout_technology_label,method_name_id,method_name_label,method_uri,sequencing_library_kit_id,sequencing_library_kit_label,sequencing_platform_id,sequencing_platform_label,sequencing_strategy_id,sequencing_strategy_label,software_counts_id,software_counts_label,software_analysis_id,software_analysis_label,score_interpretation,reference_genome_id,reference_genome_label,associated_datasets,license_label,license_id
0,zhu_2025_D4_stim48hr_cl,1,Perturb-seq,<NA>,<NA>,AAACAAGCAAACCCAAATACGTCA-1_CD4i_R2L41,chr3:179604690-179627647;1,3,3,1,ENSG00000136521,NDUFB5,protein_coding,GCGGCGGGTTTCGGTTACTG,CRISPRi,<NA>,P14DT0H0M0S,anti-CD3 antibody|anti-CD28 antibody|anti-CD2 ...,EFO:0003317|EFO:0003304|NCIT:C184729,CD4i_R2L41,D4_CE0006864,primary_cell,<NA>,Homo sapiens,lymphoid tissue,UBERON:0001744,"CD4-positive, alpha-beta T cell",CL:0000624,<NA>,<NA>,male,<NA>,adult,<NA>,healthy,<NA>,Genome-scale perturb-seq in primary human CD4+...,https://doi.org/10.64898/2025.12.23.696273,2025,Ronghui Zhu,Alexander Marson,Perturb-seq of primary human CD4-positive T ce...,\n Isolated human CD4-positive cell...,12423,2208902,EFO:0022868,endogenous,<NA>,dCas9-KRAB-Zim3,<NA>,lentivirus transduction,<NA>,lentivirus transduction,<NA>,random locus integration,<NA>,random locus integration,<NA>,constitutive transgene expression,<NA>,constitutive transgene expression,custom,<NA>,<NA>,pooled,<NA>,focused,<NA>,inhibition,Marson lab,2,2,24803,<NA>,<NA>,high-dimensional assay,<NA>,transcriptomic,<NA>,single-cell rna-seq,<NA>,Perturb-seq,<NA>,<NA>,GEM-X Flex Gene Expression Human n-plex kit,<NA>,Ultima Genomics UG100,<NA>,barcode sequencing,<NA>,CellRanger,<NA>,scanpy,<NA>,<NA>,GRCh38,"[{""dataset_accession"": ""Primary Human CD4+ T C...",MIT License,SWO:9000074
1,zhu_2025_D4_stim48hr_cl,2,Perturb-seq,<NA>,<NA>,AAACAAGCAACAAAGAAACGCCGA-1_CD4i_R2L41,chr6:2887169-2903309;-1,6,6,1,ENSG00000170542,SERPINB9,protein_coding,TCCAGGCTCGGAGCGCGGCA,CRISPRi,<NA>,P14DT0H0M0S,anti-CD3 antibody|anti-CD28 antibody|anti-CD2 ...,EFO:0003317|EFO:0003304|NCIT:C184729,CD4i_R2L41,D4_CE0006864,primary_cell,<NA>,Homo sapiens,lymphoid tissue,UBERON:0001744,"CD4-positive, alpha-beta T cell",CL:0000624,<NA>,<NA>,male,<NA>,adult,<NA>,healthy,<NA>,Genome-scale perturb-seq in primary human CD4+...,https://doi.org/10.64898/2025.12.23.696273,2025,Ronghui Zhu,Alexander Marson,Perturb-seq 

# VAR slot curation

### Standardise genes

In [31]:
cur_data.adata.var

,gene_ids,feature_types,genome,gene_name,mt
CUSTOM001_PuroR,CUSTOM001_PuroR,Gene Expression,GRCh38,PuroR,False
ENSG00000000003,ENSG00000000003,Gene Expression,GRCh38,TSPAN6,False
ENSG00000000005,ENSG00000000005,Gene Expression,GRCh38,TNMD,False
ENSG00000000419,ENSG00000000419,Gene Expression,GRCh38,DPM1,False
ENSG00000000457,ENSG00000000457,Gene Expression,GRCh38,SCYL3,False
...,...,...,...,...,...
ENSG00000291110,ENSG00000291110,Gene Expression,GRCh38,TRIM16L,False
ENSG00000291122,ENSG00000291122,Gene Expression,GRCh38,CASTOR3P,False
ENSG00000291135,ENSG00000291135,Gene Expression,GRCh38,FCGR1BP,False
ENSG00000291145,ENSG00000291145,Gene Expression,GRCh38,PPP5D1P,False


In [32]:
cur_data.standardize_genes(
    slot="var",
    input_column="gene_ids",
    input_column_type="ensembl_gene_id",
    remove_version=False,
    multiple_entries=False
)

Missing Ensembl IDs: ['ENSG00000148362', 'CUSTOM001_PuroR', 'ENSG00000139656']; attempting to fetch latest IDs...
Fetched latest Ensembl IDs: {'ENSG00000148362': 'ENSG00000310560', 'ENSG00000139656': 'ENSG00000226519'}
--------------------------------------------------
Successfully mapped 18130 out of 18130 Ensembl IDs.
--------------------------------------------------


/homes/zakirov/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [33]:
# replace gene symbols that are NOT ENSG with original provided gene symbol
cur_data.adata.var.loc[(cur_data.adata.var['gene_symbol'].isna()) &
 (~cur_data.adata.var['gene_name'].str.startswith('ENSG')), 'gene_symbol'] = cur_data.adata.var.loc[(cur_data.adata.var['gene_symbol'].isna()) &
 (~cur_data.adata.var['gene_name'].str.startswith('ENSG')), 'gene_name']

In [34]:
cur_data.adata.var.loc[cur_data.adata.var['ensembl_gene_id'] == 'CUSTOM001_PuroR', 'ensembl_gene_id'] = None

### Validate var metadata

In [35]:
cur_data.validate_data(slot='var')

2026-02-18 10:59:03,521 INFO curation_tools.curation_tools: adata.var is valid according to the var_schema.


,ensembl_gene_id,gene_symbol
index,,
0,None,PuroR
1,ENSG00000000003,TSPAN6
2,ENSG00000000005,TNMD
3,ENSG00000000419,DPM1
4,ENSG00000000457,SCYL3
...,...,...
18125,ENSG00000291110,TRIM16L
18126,ENSG00000291122,CASTOR3P
18127,ENSG00000291135,FCGR1BP


# Save the dataset

In [36]:
cur_data.save_curated_data_h5ad()

/nfs/production/mfreeberg/perturb-seq/aleks/PerturbationCatalogue/data_exploration/curation_tools/curation_tools.py:327: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  adata.obs = adata.obs.fillna(value=np.nan)
... storing 'dataset_id' as categorical
... storing 'data_modality' as categorical
... storing 'significance_criteria' as categorical
... storing 'perturbed_target_coord' as categorical
... storing 'perturbed_target_chromosome' as categorical
... storing 'perturbed_target_ensg' as categorical
... storing 'perturbed_target_symbol' as categorical
... storing 'perturbed_target_biotype' as categorical
... storing 'guide_sequence' as categorical
... storing 'perturbation_type_label' as categorical
... storing 'perturbation_type_id' as categorical
... storing 'timepoint

✅ Curated h5ad data saved to ../curated/h5ad/zhu_2025_D4_stim48hr_cl_curated.h5ad


In [37]:
cur_data.save_curated_data_parquet(split_metadata=True, save_metadata_only=True)

✅ Metadata saved to ../curated/parquet/zhu_2025_D4_stim48hr_cl_curated_metadata.parquet


# Upload to BigQuery

In [ ]:
# upload_parquet_to_bq(
#     parquet_path='../curated/parquet/zhu_2025_D1_rest_cl_curated_metadata.parquet',
#     bq_dataset_id='prj-ext-dev-pertcat-437314.perturb_seq',
#     bq_table_name='metadata',
#     key_columns=['dataset_id', 'sample_id'],
#     verbose=True
# )

# Upload to GC Storage

In [ ]:
# !gcloud storage cp ../curated/h5ad/zhu_2025_D4_stim48hr_cl_curated.h5ad gs://perturbation-catalogue-lake/perturbseq/curated/